# **Day 1 — Sprint 4 Planning, Serialization & MLOps**


### **Why Model Deployment Matters**

A trained model sitting in a notebook has no real-world value until someone besides its creator can
actually use it. Deployment is the process of taking that trained model and making it accessible as
a live, usable application — typically behind a public URL. It's also what separates a portfolio
project employers take seriously from a purely academic exercise: a model that only exists as
notebook cells and printed metrics proves you can train something, but not that you can ship it.

### **What is Serialization?**

Serialization is the process of saving a trained model (and everything it depends on) to disk in a
format that can be reloaded later — by a different script, a different machine, or a different
person entirely — without needing to retrain from scratch. For this project, that means saving three
things together: the fine-tuned DistilBERT model weights, its tokenizer (which handles all text
preprocessing), and the label mapping (which translates the model's numeric output back into human-
readable class names). Saving only the model and leaving preprocessing to be "reimplemented by hand"
later is one of the most common real-world deployment bugs — any mismatch between how text was
processed during training versus during serving silently corrupts predictions.

### **MLOps & Reproducibility**

MLOps practices ensure a model that worked in a training notebook continues to work reliably once it
leaves that notebook. Three practices matter specifically at this stage:
- **Pinned dependencies** (`requirements.txt`) — locking exact library versions so the deployment
  environment installs precisely what the model was trained and tested against, rather than
  whatever the latest versions happen to be.
- **Experiment tracking** (e.g., MLflow) — keeping every trained model version traceable back to the
  data, code, and hyperparameters that produced it.
- **Fixed seeds** — ensuring that any remaining randomness (e.g., in data splitting) produces
  consistent, reproducible results across runs.

Without these, a model that behaved correctly in development can fail to install, or behave
differently, once deployed — exactly the kind of environment instability encountered during Sprint 3
when a library upgrade silently broke the working pipeline.

## **Sprint 4 plan**

**Sprint Goal:** Deploy the DistilBERT AG News classifier as a live, public application.

**Backlog:**
1. Serialize the trained model, tokenizer, and label mapping for production use.
2. Write a reload script that reproduces a known prediction, confirming no serialization corruption.
3. Freeze a pinned `requirements.txt` for the deployment environment.
4. Build a minimal serving app (e.g., Gradio/Streamlit) around the existing `predict()` function.
5. Deploy to a public URL (e.g., Hugging Face Spaces).
6. Repository polish and final Sprint Review / Retrospective.

**Carried forward from Sprint 3 Retrospective:** Pin exact library versions in a requirements file
at the start of the sprint, and consolidate model/data reload steps into a single setup cell, to
avoid the environment-debugging time lost in Sprint 3.

In [1]:
from google.colab import drive
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import os
import json

In [2]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
final_path = "/content/drive/MyDrive/ag_news_distilbert/final_model"

tokenizer = AutoTokenizer.from_pretrained(final_path)
model = AutoModelForSequenceClassification.from_pretrained(final_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print(f"Model loaded from {final_path} onto {device}")
print("Files present:", os.listdir(final_path))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded from /content/drive/MyDrive/ag_news_distilbert/final_model onto cpu
Files present: ['config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'label_map.json']


In [4]:
print(os.listdir(final_path))

['config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'label_map.json']


### **Confirming the Saved Model Artifacts**

This step loads the DistilBERT model and tokenizer directly from Google Drive rather than retraining. Printing the folder's
contents confirms exactly what was serialized: `config.json` and `model.safetensors` (the model
architecture and trained weights), `tokenizer_config.json` and `tokenizer.json` (the tokenizer,
which is this project's "preprocessing object" — the equivalent of a scaler or encoder in a
classical ML pipeline), and `label_map.json` (added in the next step). The presence of all of these
together — not just the model weights alone — is what prevents training/serving skew once this
model is deployed.

In [5]:
label_map = {0:"World", 1:"Sports", 2:"Business", 3:"Sci/Tech"}

with open(f"{final_path}/label_map.json", "w") as f:
  json.dump(label_map, f)

print("Saved files:", os.listdir(final_path))

Saved files: ['config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'label_map.json']


### **Saving the Label Mapping**

The model itself only outputs numeric class indices (0-3) — it has no built-in concept of "World"
or "Sports." The `label_map` dictionary is what translates those indices back into human-readable
class names, and it needs to be saved and versioned alongside the model just as deliberately as the
model weights themselves. Without it, a deployed model would produce technically correct but
practically meaningless output (a bare integer) to any end user.

In [7]:
loaded_tokenizer = AutoTokenizer.from_pretrained(final_path)
loaded_model = AutoModelForSequenceClassification.from_pretrained(final_path)
with open(f"{final_path}/label_map.json", "r") as f:
  loaded_label_map = {int(k): v for k, v in json.load(f).items()}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_model.to(device)
loaded_model.eval()

def predicted(raw_text):
  inputs = loaded_tokenizer(raw_text, return_tensors="pt", truncation=True, padding=True, max_length=128)
  inputs = {k: v.to(device) for k, v in inputs.items()}
  with torch.no_grad():
    outputs = loaded_model(**inputs)
    prediction = torch.argmax(outputs.logits, dim=1).item()
  return loaded_label_map[prediction]

known_text = "The stock market rallied today after the Federal Resrve announcement."
result = predicted(known_text)
print(f"Predicted label for '{known_text}': {result}")
assert result =="Business", f"Expected 'Business', got {result} - serialization may be broken!"

print("Reload verified - serialized model reproduces expected prediction.")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Predicted label for 'The stock market rallied today after the Federal Resrve announcement.': Business
Reload verified - serialized model reproduces expected prediction.


### **Verifying the Serialized Pipeline End-to-End**

This is the most important check in this notebook: it simulates a completely fresh deployment
environment by reloading the tokenizer, model, and label map from disk into new variables rather than reusing anything already in
memory from training. If serialization had silently corrupted or mismatched any of these three
components, this is where it would surface.


 **Converting JSON keys back to integers**: JSON files always store dictionary keys as strings, so
  reloading `label_map.json` without the `{int(k): v for k, v in ...}` conversion would leave the
  keys as `"0"`, `"1"`, etc. — which would fail to match the integer output of the model, even
  though the values themselves are correct.
 **Moving tensors to the correct device**: inputs are explicitly moved to whichever device
  (`cuda` or `cpu`) the model is loaded onto, since PyTorch requires the model and its inputs to
  reside on the same device before a forward pass can run.

### **Interpreting the Result**

The known test sentence — a stock market/Federal Reserve headline — correctly predicted **Business**,
matching the expected label with no manual correction needed. This confirms the full serialized
pipeline (tokenizer → model → label map) reproduces the exact same behavior it had immediately after
training, with zero degradation or mismatch introduced by the save/reload process itself. This is
the concrete evidence that the model is safe to hand off to a separate serving application.

In [8]:
!pip freeze | grep -iE "transformers|torch|huggingface|tokenizers|numpy|scikit-learn"> requirements.txt
print(open("requirements.txt").read())

huggingface_hub==1.29.0
numpy==2.1.3
scikit-learn==1.6.1
sentence-transformers==5.7.0
tokenizers==0.23.1
torch @ https://download.pytorch.org/whl/cpu/torch-2.11.0%2Bcpu-cp313-cp313-manylinux_2_28_x86_64.whl
torchao==0.10.0
torchaudio @ https://download.pytorch.org/whl/cpu/torchaudio-2.11.0%2Bcpu-cp313-cp313-manylinux_2_28_x86_64.whl
torchcodec @ https://download.pytorch.org/whl/cpu/torchcodec-0.11.0%2Bcpu-cp313-cp313-manylinux_2_28_x86_64.whl
torchdata==0.11.0
torchsummary==1.5.1
torchtune==0.6.1
torchvision @ https://download.pytorch.org/whl/cpu/torchvision-0.26.0%2Bcpu-cp313-cp313-manylinux_2_28_x86_64.whl
transformers==5.16.1



In [9]:
!cp requirements.txt /content/drive/MyDrive/ag_news_distilbert/requirements.txt

## **Conclusion**

This notebook completed the Sprint 4 Day 1 serialization and reproducibility groundwork required
before deployment. The fine-tuned DistilBERT model, its tokenizer, and the label mapping were
confirmed saved together as a complete, self-contained artifact set — not just the model weights in
isolation. A fresh-load verification test confirmed the saved pipeline reproduces the exact same
prediction behavior it had immediately after training, with no corruption or mismatch introduced by
the save/reload process. Finally, a pinned `requirements.txt` was frozen from the current working
environment, directly addressing the library version instability encountered in Sprint 3.

With these three pieces in place — a verified serialized model, its preprocessing tokenizer, and a
pinned dependency list — the project now has everything a separate serving application needs to load
and run predictions without retraining, satisfying the core reproducibility requirements ahead of
building and deploying the live application in the remaining Sprint 4 tasks.